<a href="https://colab.research.google.com/github/Sathwik764/GenAI/blob/main/assignments/langchain/task/lang_graph_basics_assignment1_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 2

Build an ATM Machine Workflow.

State

{
    "balance":5000,
    "withdraw_amount":3000
}




   Withdraw Request

          ↓

   Enough Balance?

     /         \

    Yes          No
    ↓             ↓
   Dispense       Reject

    Cash


In [15]:
from typing_extensions import TypedDict,Literal
from langgraph.graph import StateGraph,START,END


In [14]:
#defining state
class BankState(TypedDict):
  balance:int
  withdraw:int
  message:str

In [22]:
#node 1 function - withdraw request
def withdraw_req(state:BankState) -> BankState:
  state["message"]+="user requesting to withdraw amount from bank"
  return state

#node 2 function - conditional node to check for balance available or not
def balance_check(state:BankState)-> Literal['accept','decline']:
  if state['balance']>= state['withdraw']:
    return "accept"
  else:
    return "decline"

#node 2 function - withdraw accepted and successful
def req_accept(state: BankState) -> BankState:
    state['balance'] = state['balance'] - state['withdraw']
    state['message'] = "Withdrawal of {} successful. Remaining balance: {}".format(state['withdraw'], state['balance'])
    return state

#node 3 function - withdraw rejected and declined
def req_decline(state:BankState) -> BankState:
  state['message'] = "Withdrawal of {} unsuccessful due to insufficient balance: {}".format(state['withdraw'], state['balance'])
  return state

In [23]:
system=StateGraph(BankState)
system.add_node("request",withdraw_req)
system.add_node("accept",req_accept)
system.add_node("decline",req_decline)

system.add_edge(START,"request")
system.add_conditional_edges(
    "request",balance_check,{
        "accept":"accept",
        "decline":"decline"
    }
)
system.add_edge("accept",END)
system.add_edge("decline",END)

graph=system.compile()

In [25]:
initial_state = {
    "balance": 5000,
    "withdraw_amount": 3000,
    "message": ""
}

#user input
withdraw=int(input("enter amount to be withdrawn"))
balance=5000

initial_state: BankState = BankState(
  balance=balance,
  withdraw=withdraw,
  message=""
)

result=graph.invoke(initial_state)
print(result['message'])

enter amount to be withdrawn2500
Withdrawal of 2500 successful. Remaining balance: 2500
